In [52]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.decomposition import PCA
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler
import partial_r as pr
from figure_formatting import setup_figure, save_figure
import importlib
importlib.reload(pr)

<module 'partial_r' from '/mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/code/partial_r.py'>

### Load and prep data

### Outputs from `Step3_abcd_prep_harmonized_dfs`

In [2]:
# directories
# ROOT_DIR = Path("/Users/bmacedo/Desktop/final_WM")
ROOT_DIR = Path("/mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome")
RESULTS_DIR = ROOT_DIR / "output_data" / "results"

# ---- Load datasets ----
dfA_macro = pd.read_csv(RESULTS_DIR / "dfA_macro_12_10.csv")
dfB_macro = pd.read_csv(RESULTS_DIR / "dfB_macro_12_10.csv")

dfA_NODDI = pd.read_csv(RESULTS_DIR / "dfA_NODDI_12_10.csv")
dfB_NODDI = pd.read_csv(RESULTS_DIR / "dfB_NODDI_12_10.csv")

dfA_macro_plus_NODDI = pd.read_csv(RESULTS_DIR / "dfA_macro_plus_NODDI_12_10.csv")
dfB_macro_plus_NODDI = pd.read_csv(RESULTS_DIR / "dfB_macro_plus_NODDI_12_10.csv")

dfA_DKI = pd.read_csv(RESULTS_DIR / "dfA_DKI_12_10.csv")
dfB_DKI = pd.read_csv(RESULTS_DIR / "dfB_DKI_12_10.csv")

dfA_macro_plus_DKI = pd.read_csv(RESULTS_DIR / "dfA_macro_plus_DKI_12_10.csv")
dfB_macro_plus_DKI = pd.read_csv(RESULTS_DIR / "dfB_macro_plus_DKI_12_10.csv")

# ---- Logging ----
print("[LOAD macro] A:", dfA_macro.shape, "| B:", dfB_macro.shape)
print("[LOAD NODDI] A:", dfA_NODDI.shape, "| B:", dfB_NODDI.shape)
print("[LOAD macro+NODDI] A:", dfA_macro_plus_NODDI.shape, "| B:", dfB_macro_plus_NODDI.shape)
print("[LOAD DKI] A:", dfA_DKI.shape, "| B:", dfB_DKI.shape)
print("[LOAD macro+DKI] A:", dfA_macro_plus_DKI.shape, "| B:", dfB_macro_plus_DKI.shape)

[LOAD macro] A: (8175, 1135) | B: (8175, 1135)
[LOAD NODDI] A: (8175, 391) | B: (8175, 391)
[LOAD macro+NODDI] A: (8175, 1507) | B: (8175, 1507)
[LOAD DKI] A: (8175, 1135) | B: (8175, 1135)
[LOAD macro+DKI] A: (8175, 2251) | B: (8175, 2251)


These dataframes include the full set of participants (both split halves), harmonized using either the discovery or replication split half.

Here, we only take the split half we want for each analysis. E.g. for the discovery analyses, we take only the discovery participants from the dataframe harmonized on the discovery data.

In [3]:
dfA_macro = dfA_macro[dfA_macro["matched_group"] == 1].copy()
dfB_macro = dfB_macro[dfB_macro["matched_group"] == 2].copy()

dfA_NODDI = dfA_NODDI[dfA_NODDI["matched_group"] == 1].copy()
dfB_NODDI = dfB_NODDI[dfB_NODDI["matched_group"] == 2].copy()

dfA_macro_plus_NODDI = dfA_macro_plus_NODDI[dfA_macro_plus_NODDI["matched_group"] == 1].copy()
dfB_macro_plus_NODDI = dfB_macro_plus_NODDI[dfB_macro_plus_NODDI["matched_group"] == 2].copy()

dfA_DKI = dfA_DKI[dfA_DKI["matched_group"] == 1].copy()
dfB_DKI = dfB_DKI[dfB_DKI["matched_group"] == 2].copy()

dfA_macro_plus_DKI = dfA_macro_plus_DKI[dfA_macro_plus_DKI["matched_group"] == 1].copy()
dfB_macro_plus_DKI = dfB_macro_plus_DKI[dfB_macro_plus_DKI["matched_group"] == 2].copy()

### Helper functions for downstream analyses

In [34]:
PREFIXES = ['Association_', 'Cerebellum_', 'Commissure_', 'ProjectionBrainstem_', 'ProjectionBasalGanglia_']
STOPWORDS = {"of", "and", "in", "for", "to", "with", "on", "at"}

# regexes used repeatedly
RE_CAMEL_SPACE = re.compile(r'(?<!^)(?=[A-Z])') # capital letter that is not the first character in the string
RE_LR_END = re.compile(r'(L|R)$') # l/r at end of string
RE_MSMT_PREFIX = re.compile(r'^bundle_') # msmt prefix
RE_MM_SUFFIX = re.compile(r'\s*mm[23]?\s*$', flags=re.IGNORECASE) # trailing millimeter units e..g mm, mm2, mm3

def strip_known_prefix(name: str, prefixes=PREFIXES) -> str:
    for p in prefixes:
        if name.startswith(p): 
            return name[len(p):], p # returns stripped name and prefix/bundle type
    return name, None

def clean_bundle_name(name: str, strip_lr: bool = False) -> str:
    name = str(name)
    if strip_lr: name = RE_LR_END.sub('', name)
    name, _ = strip_known_prefix(name)

    # adds spaces between capital letters e.g. InferiorLongitudinalFasciculus -> Inferior Longitudinal Fasciculus
    name = RE_CAMEL_SPACE.sub(' ', name)
    return name.strip()

def bundle_type_from_name(bundle: str) -> str:
    bundle = str(bundle)
    _, p = strip_known_prefix(bundle)
    if p is None: return "Other"
    return p

def clean_metrics_name(name: str) -> str:
    name = str(name).replace('_', ' ')
    name = RE_MM_SUFFIX.sub('', name).strip() # remove mm units

    upper = name.upper()
    if upper.startswith("NODDI") or upper.startswith("DKI"):
        return " ".join(w.upper() for w in name.split())

    ordinal_allowed = "quarter" in name.lower()

    def ordinalize(token: str) -> str:
        n = int(token)
        if n % 10 == 1 and n % 100 != 11: return f"{n}st"
        if n % 10 == 2 and n % 100 != 12: return f"{n}nd"
        if n % 10 == 3 and n % 100 != 13: return f"{n}rd"
        return f"{n}th"

    cleaned = []
    for w in name.split():
        if w.isdigit():
            cleaned.append(ordinalize(w) if ordinal_allowed else w)
        elif w.lower() in STOPWORDS:
            cleaned.append(w.lower())
        else:
            cleaned.append(w.capitalize())
    return " ".join(cleaned)

def parse_for_order_and_plot(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    clean = out["feature"].astype(str).str.replace(RE_MSMT_PREFIX, "", regex=True)

    parts = clean.str.split("_")

    out["bundle_raw"] = parts.str[:2].str.join("_")
    out["metric_raw"] = parts.str[2:].str.join("_")

    out = out.dropna(subset=["bundle_raw", "metric_raw"])
    out = out[out["metric_raw"] != ""]

    out["bundle_type"] = out["bundle_raw"].map(bundle_type_from_name)
    out["bundle"] = out["bundle_raw"].map(lambda s: clean_bundle_name(s, strip_lr=True))

    out["metric_raw"] = out["metric_raw"].astype(str)
    out["metric_clean"] = out["metric_raw"]

    return out

def build_tidy(df: pd.DataFrame, split_label: str) -> pd.DataFrame:
    tidy = df[["bundle", "bundle_type", "metric_raw", "metric_clean", "partial_r"]].copy()
    tidy = tidy.rename(columns={"metric_clean": "metric"})
    tidy["split"] = split_label
    return tidy

In [5]:
tracts = sorted(set(
    "_".join(c.split("_")[1:3])
    for c in dfA_macro_plus_NODDI.columns
    if c.startswith("bundle_")
))

print(len(tracts))

62


In [6]:
tract_dict = {t: clean_bundle_name(t) for t in tracts}

In [7]:
# --- cleanup helper: drop mean columns, strip _median from remaining columns ---
def clean_bundle_columns(df):
    df = df.copy()

    bundle_cols_mean = [c for c in df.columns if c.startswith("bundle_") and "_mean" in c]
    df = df.drop(columns=bundle_cols_mean, errors="ignore")

    df = df.rename(columns=lambda c: c[:-len("_median")]
        if c.startswith("bundle_") and c.endswith("_median")
        else c)

    return df

dfs_to_clean = ["dfA_macro", "dfB_macro", "dfA_NODDI", "dfB_NODDI", "dfA_macro_plus_NODDI", "dfB_macro_plus_NODDI",
                "dfA_DKI", "dfB_DKI", "dfA_macro_plus_DKI", "dfB_macro_plus_DKI"]

for df_name in dfs_to_clean:
    globals()[df_name] = clean_bundle_columns(globals()[df_name])

# --- helper to extract metric name from cleaned columns ---
def extract_metric_name(col):
    if not col.startswith("bundle_"):
        return None

    parts = col[len("bundle_"):].split("_")
    if len(parts) < 3: return None

    # first two parts are the bundle name, rest is the metric
    return "_".join(parts[2:])


# --- metric summaries after cleaning ---
noddi_macro_metrics = sorted({m for c in dfA_macro_plus_NODDI.columns if (m := extract_metric_name(c)) is not None})
dki_macro_metrics = sorted({m for c in dfA_macro_plus_DKI.columns if (m := extract_metric_name(c)) is not None})

print(f"NODDI metrics count: {len(noddi_macro_metrics)}")
print(f"DKI metrics count: {len(dki_macro_metrics)}")

print("\nNODDI metrics:")
print(noddi_macro_metrics)

print("\nDKI metrics:")
print(dki_macro_metrics)

NODDI metrics count: 20
DKI metrics count: 26

NODDI metrics:
['1st_quarter_volume_mm3', '2nd_and_3rd_quarter_volume_mm3', '4th_quarter_volume_mm3', 'NODDI_icvf', 'NODDI_isovf', 'NODDI_od', 'area_of_end_region_1_mm2', 'area_of_end_region_2_mm2', 'curl', 'elongation', 'irregularity', 'radius_of_end_region_1_mm', 'radius_of_end_region_2_mm', 'span_mm', 'total_area_of_end_regions_mm2', 'total_radius_of_end_regions_mm', 'total_surface_area_mm2', 'total_volume_mm3', 'volume_of_end_branches_1', 'volume_of_end_branches_2']

DKI metrics:
['1st_quarter_volume_mm3', '2nd_and_3rd_quarter_volume_mm3', '4th_quarter_volume_mm3', 'DKI_ad', 'DKI_ak', 'DKI_fa', 'DKI_kfa', 'DKI_md', 'DKI_mk', 'DKI_mkt', 'DKI_rd', 'DKI_rk', 'area_of_end_region_1_mm2', 'area_of_end_region_2_mm2', 'curl', 'elongation', 'irregularity', 'radius_of_end_region_1_mm', 'radius_of_end_region_2_mm', 'span_mm', 'total_area_of_end_regions_mm2', 'total_radius_of_end_regions_mm', 'total_surface_area_mm2', 'total_volume_mm3', 'volume_o

In [8]:
noddi_macro_metrics_dict = {m: clean_metrics_name(m) for m in noddi_macro_metrics}
dki_macro_metrics_dict = {m: clean_metrics_name(m) for m in dki_macro_metrics}

### Run Partial R analysis

In [9]:
covariates = ["age", "sex", "t1post_dwi_contrast",
              "General_SES", "School", "Family_Values", "Family_Turmoil", "Dense_Urban_Poverty", "Extracurriculars", "Screen_Time"]

scenarios = [("NODDI", False, dfA_macro_plus_NODDI, dfB_macro_plus_NODDI),
             ("DKI", False, dfA_macro_plus_DKI, dfB_macro_plus_DKI),
             ("NODDI", True,  dfA_macro_plus_NODDI, dfB_macro_plus_NODDI)]

partial_r_results = {}

for name, include_etiv, df_A, df_B in scenarios:
    msmt_cols = [c for c in df_A.columns if c.startswith("bundle_")]
    print(f"Running partial r models for {name}, include_etiv={include_etiv}")

    partial_r_results[(name, include_etiv, "A")] = pr.run_partial_r(df_A, msmt_cols, covariates, include_etiv, target_cov="General_SES")
    partial_r_results[(name, include_etiv, "B")] = pr.run_partial_r(df_B, msmt_cols, covariates, include_etiv, target_cov="General_SES")

Running partial r models for NODDI, include_etiv=False
Running partial r models for DKI, include_etiv=False
Running partial r models for NODDI, include_etiv=True


### Save partial r values to compare to mutlivariate haufe weights for supp fig 9

In [10]:
scenario_name, scenario_etiv = "NODDI", False
dfA_r = partial_r_results[(scenario_name, scenario_etiv, "A")].copy()
dfB_r = partial_r_results[(scenario_name, scenario_etiv, "B")].copy()

# Directory
SUPPFIG9_DIR = ROOT_DIR / "output_data" / "results" / "supplemental_figures" / "suppfig9"
SUPPFIG9_DIR.mkdir(parents=True, exist_ok=True)

# save partial rs to load into multivariate notebook and compare to haufe weights 
# add group label
dfA_r["group"] = "A"
dfB_r["group"] = "B"

# keep only the SES row
dfA_r_clean = dfA_r[dfA_r["covariate"] == "General_SES"].copy()
dfB_r_clean = dfB_r[dfB_r["covariate"] == "General_SES"].copy()

partial_r_long = pd.concat([dfA_r_clean, dfB_r_clean], axis=0, ignore_index=True)

# output path
out_csv = SUPPFIG9_DIR / "partial_r_General_SES_NODDI_etivFalse.csv"

# save
partial_r_long.to_csv(out_csv, index=False)

print(f"[SAVE] partial_r → {out_csv} | shape: {partial_r_long.shape}")

[SAVE] partial_r → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig9/partial_r_General_SES_NODDI_etivFalse.csv | shape: (2480, 7)


### Save partial r values to compare to HBN partial rs for Figure 6

In [11]:
FIG6_DIR = ROOT_DIR / "output_data" / "results" / "main_figures" / "figure6"
FIG6_DIR.mkdir(parents=True, exist_ok=True)

def average_split_halves(dfA_r, dfB_r, id_cols, val_col="partial_r"):
    a = (dfA_r.groupby(id_cols, as_index=False)[val_col].mean().rename(columns={val_col: "partial_r_A"}))
    b = (dfB_r.groupby(id_cols, as_index=False)[val_col].mean().rename(columns={val_col: "partial_r_B"}))

    m = a.merge(b, on=id_cols, how="inner")
    m["partial_r_avg"] = m[["partial_r_A", "partial_r_B"]].mean(axis=1)

    return m

dfAB = average_split_halves(dfA_r, dfB_r, id_cols="feature", val_col="partial_r")
dfAB.head()

out_path = FIG6_DIR / "abcd_partialr_avg.csv"

dfAB.to_csv(out_path, index=False)

print(f"[SAVE] → {out_path} | shape: {dfAB.shape}")

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure6/abcd_partialr_avg.csv | shape: (1240, 4)


### Figure 2 Plotting

#### Figure 2a

In [25]:
# Directories
MAIN_FIG2_DIR = ROOT_DIR / "output_data" / "results" / "main_figures" / "figure2"
SUPPFIG1_DIR = ROOT_DIR / "output_data" / "results" / "supplemental_figures" / "suppfig1"
SUPPFIG2_DIR = ROOT_DIR / "output_data" / "results" / "supplemental_figures" / "suppfig2"

for d in [MAIN_FIG2_DIR, SUPPFIG1_DIR, SUPPFIG2_DIR]:
    d.mkdir(parents=True, exist_ok=True)

scenarios = [
    ("NODDI", False),   # MAIN FIGURE
    ("DKI", False),     # DKI SUPPLEMENT
    ("NODDI", True),    # TBV SUPPLEMENT
]

hue_order =  ["Association_", "ProjectionBasalGanglia_", "ProjectionBrainstem_", "Cerebellum_", "Commissure_"]

palette_bundle = {
  "Association_"          : "#4E79A7",
  "ProjectionBrainstem_"  : "#F28E2B",
  "ProjectionBasalGanglia_" : "#59A14F",
  "Cerebellum_"           : "#E15759",
  "Commissure_"           : "#B07AA1"
}

bundle_label_map = {
    "Association_": "Association",
    "ProjectionBasalGanglia_": "Projection (Basal Ganglia)",
    "ProjectionBrainstem_": "Projection (Brainstem)",
    "Cerebellum_": "Cerebellum",
    "Commissure_": "Corpus Callosum",
}

for scenario_name, scenario_etiv in scenarios:

    print(f"\n[SCENARIO] {scenario_name} | eTIV={scenario_etiv}")

    # Load data
    dfA_r = partial_r_results[(scenario_name, scenario_etiv, "A")].copy()
    dfB_r = partial_r_results[(scenario_name, scenario_etiv, "B")].copy()

    print(f"[LOAD] dfA_r: {dfA_r.shape} | dfB_r: {dfB_r.shape}")

    if scenario_name == "NODDI" and scenario_etiv is False:
        metrics_kwargs = {"noddi_macro_metrics_dict": noddi_macro_metrics_dict}
        save_dir = MAIN_FIG2_DIR
        file_stub = "FIG2_heatmap"

    elif scenario_name == "DKI" and scenario_etiv is False:
        metrics_kwargs = {"dki_macro_metrics_dict": dki_macro_metrics_dict}
        save_dir = SUPPFIG1_DIR
        file_stub = "SUPP_heatmap_DKI_etivFalse"

    elif scenario_name == "NODDI" and scenario_etiv is True:
        metrics_kwargs = {"noddi_macro_metrics_dict": noddi_macro_metrics_dict}
        save_dir = SUPPFIG2_DIR
        file_stub = "SUPP_heatmap_NODDI_etivTrue"

    # Plot
    sns.set_style("white")
    cmap = sns.diverging_palette(250, 15, s=95, l=55, as_cmap=True)

    fig, ax = setup_figure(width_mm=180, height_mm=130, margins_mm=(40, 30, 18, 6))

    vals = np.concatenate([dfA_r["partial_r"], dfB_r["partial_r"]])
    vmax = np.nanpercentile(np.abs(vals), 98)
    norm = plt.Normalize(vmin=-vmax, vmax=vmax)

    pr.make_partial_r_heatmap_split_half(dfA_r, dfB_r, "General_SES", ax, cmap, norm=norm, bundle_palette=palette_bundle, **metrics_kwargs)

    ax.set_title("")
    ax.tick_params(axis="x", labelbottom=True)
    ax.tick_params(axis="y", labelleft=True)

    fig.canvas.draw()

    # Save
    svg_path = save_dir / f"{file_stub}.svg"
    png_path = save_dir / f"{file_stub}.png"

    save_figure(fig, svg_path, autofit=False)
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    print(f"[SAVE] SVG: {svg_path}")
    print(f"[SAVE] PNG: {png_path}")

    plt.close(fig)


[SCENARIO] NODDI | eTIV=False
[LOAD] dfA_r: (1240, 6) | dfB_r: (1240, 6)


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] SVG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure2/FIG2_heatmap.svg
[SAVE] PNG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure2/FIG2_heatmap.png

[SCENARIO] DKI | eTIV=False
[LOAD] dfA_r: (1612, 6) | dfB_r: (1612, 6)


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] SVG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig1/SUPP_heatmap_DKI_etivFalse.svg
[SAVE] PNG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig1/SUPP_heatmap_DKI_etivFalse.png

[SCENARIO] NODDI | eTIV=True
[LOAD] dfA_r: (1240, 6) | dfB_r: (1240, 6)


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] SVG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig2/SUPP_heatmap_NODDI_etivTrue.svg
[SAVE] PNG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig2/SUPP_heatmap_NODDI_etivTrue.png


#### Figure 2b

In [46]:
for scenario_name, scenario_etiv in scenarios:
    print(f"\n[SCENARIO] {scenario_name} | eTIV={scenario_etiv}")
    dfA_r = partial_r_results[(scenario_name, scenario_etiv, "A")].copy()
    dfB_r = partial_r_results[(scenario_name, scenario_etiv, "B")].copy()

    if scenario_name == "NODDI" and scenario_etiv is False:
        primary_prefix, save_dir, file_stub = "NODDI", MAIN_FIG2_DIR, "FIG2_violin"
    elif scenario_name == "DKI" and scenario_etiv is False:
        primary_prefix, save_dir, file_stub = "DKI", SUPPFIG1_DIR, "SUPPFIG1_violin"
    elif scenario_name == "NODDI" and scenario_etiv is True:
        primary_prefix, save_dir, file_stub = "NODDI", SUPPFIG2_DIR, "SUPPFIG2_violin"

    fig, ax = setup_figure(width_mm=126, height_mm=60, margins_mm=(14, 40, 22, 0), axes_linewidth=0.8)

    prefixes = ['Association_', 'Cerebellum_', 'Commissure_', 'ProjectionBrainstem_', 'ProjectionBasalGanglia_']

    hue_order = [
        "Association_",
        "ProjectionBasalGanglia_",
        "ProjectionBrainstem_",
        "Cerebellum_",
        "Commissure_",
    ]

    A_parsed = parse_for_order_and_plot(dfA_r)
    B_parsed = parse_for_order_and_plot(dfB_r)
    tidy = pd.concat([build_tidy(A_parsed, "Split-half A (Discovery)"),
                      build_tidy(B_parsed, "Split-half B (Replication)")], ignore_index=True)
    # tidy["bundle_type"] = tidy["bundle"].apply(assign_bundle_type)

    macro_keep_raw = {"total_volume_mm3", "total_area_of_end_regions_mm2", "total_surface_area_mm2",
                      "irregularity", "curl", "mean_length_mm", "span_mm"}
    tidy["is_primary"] = tidy["metric_raw"].str.startswith(primary_prefix)
    tidy = tidy[tidy["is_primary"] | tidy["metric_raw"].isin(macro_keep_raw)].copy()

    eff = tidy.groupby(["metric_raw", "metric"])["partial_r"].apply(lambda s: s.mean()).reset_index(name="eff")
    primary_sorted = eff[eff["metric_raw"].str.startswith(primary_prefix)].sort_values("eff")["metric"].tolist()
    macro_sorted = eff[eff["metric_raw"].isin(macro_keep_raw)].sort_values("eff")["metric"].tolist()
    metric_order = primary_sorted + macro_sorted

    n_primary, n_metrics = len(primary_sorted), len(metric_order)
    tidy["metric"] = pd.Categorical(tidy["metric"], categories=metric_order, ordered=True)
    
    tidy["is_primary"] = tidy["metric_raw"].str.startswith(primary_prefix)
    tidy = tidy[tidy["is_primary"] | tidy["metric_raw"].isin(macro_keep_raw)].copy()
    
    ax.set_xlim(-0.5, n_metrics - 0.5)
    ax.axvspan(-0.5, n_primary - 0.5, color="1.0", zorder=0)
    ax.axvline(n_primary - 0.5, color="0.25", lw=0.7)

    split_order = ["Split-half A (Discovery)", "Split-half B (Replication)"]
    split_label_map = {"Split-half A (Discovery)": "Discovery", "Split-half B (Replication)": "Replication"}
    violin_palette = {split_order[0]: (0.96, 0.96, 0.96, 1.0), split_order[1]: (0.80, 0.80, 0.80, 1.0)}

    n_coll_before = len(ax.collections)
    sns.violinplot(data=tidy, x="metric", y="partial_r", hue="split", hue_order=split_order,
                   dodge=True, width=0.85, inner=None, cut=0, linewidth=0.8,
                   palette=violin_palette, ax=ax)
    fig.canvas.draw()

    for coll in ax.collections[n_coll_before:]:
        coll.set_alpha(1.0)
        coll.set_edgecolor((0.30, 0.30, 0.30, 1.0))
        coll.set_linewidth(0.8)

    leg = ax.get_legend()
    if leg is not None: leg.remove()

    violin_cols = [c for c in ax.collections[n_coll_before:] if hasattr(c, "get_paths") and len(c.get_paths()) > 0]

    centers_A, centers_B = np.empty(n_metrics, float), np.empty(n_metrics, float)
    k = 0
    for j in range(n_metrics):
        pA = violin_cols[k].get_paths()[0]; xsA = pA.vertices[:, 0]; centers_A[j] = 0.5*(xsA.min()+xsA.max()); k += 1
        pB = violin_cols[k].get_paths()[0]; xsB = pB.vertices[:, 0]; centers_B[j] = 0.5*(xsB.min()+xsB.max()); k += 1
    midpoints = 0.5*(centers_A + centers_B)

    x_index = {m: i for i, m in enumerate(metric_order)}
    rng = np.random.default_rng(0)

    point_alpha, point_size, jitter = 0.8, 6, 0.055

    for split in split_order:
        sub = tidy[tidy["split"] == split].copy()
        idx = sub["metric"].map(x_index).to_numpy(int)
        centers = centers_A[idx] if split == split_order[0] else centers_B[idx]
        xs = centers + rng.uniform(-jitter, jitter, len(sub))
        ys = sub["partial_r"].to_numpy(float)
        bt_arr = sub["bundle_type"].to_numpy()
        for bt in hue_order:
            m = (bt_arr == bt)
            ax.scatter(xs[m], ys[m], s=point_size, c=[palette_bundle[bt]],
                       alpha=point_alpha, linewidths=0, zorder=5)

    ymin, ymax = tidy["partial_r"].min(), tidy["partial_r"].max()
    pad = 0.08*(ymax - ymin if ymax > ymin else 1)
    ax.set_ylim(ymin - pad, ymax + pad)

    ax.yaxis.grid(True, color="0.96", linewidth=0.6)
    ax.axhline(0, color="0.35", lw=0.55)
    for spine in ax.spines.values(): spine.set_linewidth(0.6)
    ax.tick_params(width=0.6, length=3)

    ax.set_xticks(midpoints)

    ax.set_xticklabels([clean_metrics_name(m) for m in metric_order], rotation=35, ha="right")

    sns.despine(ax=ax)
    ax.set_ylabel(r"Partial $r$ (General Exposome)")
    ax.set_xlabel("")

    split_handles = [Patch(facecolor=violin_palette[s], edgecolor=(0.30,0.30,0.30,1.0),
                           label=split_label_map[s]) for s in split_order]
    
    bundle_handles = [
        Line2D(
            [0], [0],
            marker='o',
            linestyle='',
            markerfacecolor=palette_bundle[b],
            markeredgecolor='none',
            markersize=5.5,
            label=bundle_label_map[b]
        )
        for b in hue_order
    ]

    fig.legend(
        handles=split_handles,
        title="Split",
        loc="upper left",
        bbox_to_anchor=(0.7, 0.98),
        frameon=False
    )
    
    fig.legend(
        handles=bundle_handles,
        title="Bundle Type",
        loc="upper left",
        bbox_to_anchor=(0.7, 0.76),
        frameon=False
    )

    svg_path = save_dir / f"{file_stub}.svg"
    png_path = save_dir / f"{file_stub}.png"
    save_figure(fig, svg_path, autofit=False)
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    print(f"[SAVE] → {svg_path}")
    print(f"[SAVE] → {png_path}")
    plt.close(fig)


[SCENARIO] NODDI | eTIV=False


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure2/FIG2_violin.svg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure2/FIG2_violin.png

[SCENARIO] DKI | eTIV=False


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig1/SUPPFIG1_violin.svg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig1/SUPPFIG1_violin.png

[SCENARIO] NODDI | eTIV=True


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig2/SUPPFIG2_violin.svg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig2/SUPPFIG2_violin.png


#### Figure 2c

In [14]:
for scenario_name, scenario_etiv in scenarios:
    print(f"\n[SCENARIO] {scenario_name} | eTIV={scenario_etiv}")
    dfA_r = partial_r_results[(scenario_name, scenario_etiv, "A")].copy()
    dfB_r = partial_r_results[(scenario_name, scenario_etiv, "B")].copy()

    if scenario_name == "NODDI" and scenario_etiv is False:
        save_dir, file_stub = MAIN_FIG2_DIR, "FIG2_concordance"
    elif scenario_name == "DKI" and scenario_etiv is False:
        save_dir, file_stub = SUPPFIG1_DIR, "SUPPFIG1_concordance"
    elif scenario_name == "NODDI" and scenario_etiv is True:
        save_dir, file_stub = SUPPFIG2_DIR, "SUPPFIG2_concordance"

    fig, ax = setup_figure(width_mm=51, height_mm=60, margins_mm=(15, 0, 20, 6))

    disc_color, repl_color = "#0033A0", "#7BA6E6"
    alpha = 0.4

    rA_all = dfA_r["partial_r"].to_numpy()
    rB_all = dfB_r["partial_r"].to_numpy()
    r_concord = np.corrcoef(rA_all, rB_all)[0, 1]

    ax.scatter(rA_all, rB_all, s=12, color=repl_color, alpha=alpha, edgecolor="none")

    lo = float(min(rA_all.min(), rB_all.min()))
    hi = float(max(rA_all.max(), rB_all.max()))
    ax.plot([lo, hi], [lo, hi], color="black", lw=1.0, ls="--")

    slope, intercept = np.polyfit(rA_all, rB_all, 1)
    xs = np.linspace(lo, hi, 300)
    ax.plot(xs, slope * xs + intercept, color=disc_color, lw=2)

    ax.text(0.05, 0.95, f"$r$ = {r_concord:.2f}", fontsize=7,
            transform=ax.transAxes, ha="left", va="top")

    ax.set_xlabel("Discovery Partial $r$")
    ax.set_ylabel("Replication Partial $r$")

    sns.despine(ax=ax)

    svg_path = save_dir / f"{file_stub}.svg"
    png_path = save_dir / f"{file_stub}.png"
    save_figure(fig, svg_path, autofit=False)
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    print(f"[SAVE] → {svg_path}")
    print(f"[SAVE] → {png_path}")
    plt.close(fig)

findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica



[SCENARIO] NODDI | eTIV=False


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure2/FIG2_concordance.svg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure2/FIG2_concordance.png

[SCENARIO] DKI | eTIV=False


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig1/SUPPFIG1_concordance.svg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig1/SUPPFIG1_concordance.png

[SCENARIO] NODDI | eTIV=True


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig2/SUPPFIG2_concordance.svg
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/suppfig2/SUPPFIG2_concordance.png


### Sex x Exposome interaction model

In [50]:
covariates = [
    "age", "sex", "t1post_dwi_contrast",
    "General_SES", "School", "Family_Values", "Family_Turmoil",
    "Dense_Urban_Poverty", "Extracurriculars", "Screen_Time"
]

scenarios = [
    ("NODDI", False, dfA_macro_plus_NODDI, dfB_macro_plus_NODDI),
    ("DKI",   False, dfA_macro_plus_DKI,   dfB_macro_plus_DKI),
    ("NODDI", True,  dfA_macro_plus_NODDI, dfB_macro_plus_NODDI),
]

interaction_results = {}

for name, include_etiv, df_A, df_B in scenarios:
    msmt_cols = [c for c in df_A.columns if c.startswith("bundle_")]
    print(f"Running sex x SES interaction models for {name}, include_etiv={include_etiv}")

    interaction_results[(name, include_etiv, "A")] = run_interaction_tstats(
        df_A,
        msmt_cols,
        covariates,
        exposure="General_SES",
        sex_var="sex",
        include_etiv=include_etiv
    )

    interaction_results[(name, include_etiv, "B")] = run_interaction_tstats(
        df_B,
        msmt_cols,
        covariates,
        exposure="General_SES",
        sex_var="sex",
        include_etiv=include_etiv
    )

Running sex x SES interaction models for NODDI, include_etiv=False
Running sex x SES interaction models for DKI, include_etiv=False
Running sex x SES interaction models for NODDI, include_etiv=True


In [54]:
SUPP_INTERACTIONS_DIR = ROOT_DIR / "output_data" / "results" / "supplemental_figures" / "supp_interactions"
SUPP_INTERACTIONS_DIR.mkdir(parents=True, exist_ok=True)

scenarios = [
    ("NODDI", False),
    ("DKI", False),
    ("NODDI", True),
]

palette_bundle = {
    "Association_"            : "#4E79A7",
    "ProjectionBrainstem_"    : "#F28E2B",
    "ProjectionBasalGanglia_" : "#59A14F",
    "Cerebellum_"             : "#E15759",
    "Commissure_"             : "#B07AA1"
}

for scenario_name, scenario_etiv in scenarios:

    print(f"\n[SCENARIO] {scenario_name} | eTIV={scenario_etiv}")

    dfA_t = interaction_results[(scenario_name, scenario_etiv, "A")].copy()
    dfB_t = interaction_results[(scenario_name, scenario_etiv, "B")].copy()

    if scenario_name == "NODDI" and scenario_etiv is False:
        metrics_kwargs = {"noddi_macro_metrics_dict": noddi_macro_metrics_dict}
        save_dir = SUPP_INTERACTIONS_DIR
        file_stub = "SUPPFIG_sexXSES_interaction_NODDI"

    elif scenario_name == "DKI" and scenario_etiv is False:
        metrics_kwargs = {"dki_macro_metrics_dict": dki_macro_metrics_dict}
        save_dir = SUPP_INTERACTIONS_DIR
        file_stub = "SUPPFIG_sexXSES_interaction_DKI"

    elif scenario_name == "NODDI" and scenario_etiv is True:
        metrics_kwargs = {"noddi_macro_metrics_dict": noddi_macro_metrics_dict}
        save_dir = SUPP_INTERACTIONS_DIR
        file_stub = "SUPPFIG_sexXSES_interaction_NODDI_etivTrue"

    sns.set_style("white")
    cmap = sns.diverging_palette(250, 15, s=95, l=55, as_cmap=True)

    fig, ax = setup_figure(width_mm=180, height_mm=130, margins_mm=(40, 30, 18, 6))

    vals = np.concatenate([dfA_t["t_stat"].values, dfB_t["t_stat"].values])
    vals = vals[~np.isnan(vals)]
    vmax = np.nanpercentile(np.abs(vals), 98) if len(vals) else 1.0
    if vmax == 0:
        vmax = 1.0
    norm = plt.Normalize(vmin=-vmax, vmax=vmax)

    pr.make_interaction_t_heatmap_split_half(
        dfA_t,
        dfB_t,
        covariate="General_SES",
        ax=ax,
        cmap=cmap,
        norm=norm,
        bundle_palette=palette_bundle,
        use_fdr=True,
        alpha=0.05,
        sig_mode="both",
        cbar_label="Sex × General SES interaction (t-statistic)",
        **metrics_kwargs
    )

    ax.set_title("")
    ax.tick_params(axis="x", labelbottom=True)
    ax.tick_params(axis="y", labelleft=True)

    fig.canvas.draw()

    svg_path = save_dir / f"{file_stub}.svg"
    png_path = save_dir / f"{file_stub}.png"

    save_figure(fig, svg_path, autofit=False)
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    print(f"[SAVE] SVG: {svg_path}")
    print(f"[SAVE] PNG: {png_path}")

    plt.close(fig)


[SCENARIO] NODDI | eTIV=False


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] SVG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/supp_interactions/SUPPFIG_sexXSES_interaction_NODDI.svg
[SAVE] PNG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/supp_interactions/SUPPFIG_sexXSES_interaction_NODDI.png

[SCENARIO] DKI | eTIV=False


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] SVG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/supp_interactions/SUPPFIG_sexXSES_interaction_DKI.svg
[SAVE] PNG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/supp_interactions/SUPPFIG_sexXSES_interaction_DKI.png

[SCENARIO] NODDI | eTIV=True


findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic family 'sans-serif' not found because none of the following families were found: Helvetica
findfont: Generic f

[SAVE] SVG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/supp_interactions/SUPPFIG_sexXSES_interaction_NODDI_etivTrue.svg
[SAVE] PNG: /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/supplemental_figures/supp_interactions/SUPPFIG_sexXSES_interaction_NODDI_etivTrue.png


### Save mass univariate data for use in Figure 3 notebook

### Principal Component Analysis of partial-r scores across mass-univairate bundle x metric models.

In [18]:
def keep_lr(df):
    df = df.copy()
    parts = df["feature"].str.split("_")
    df["bundle"] = parts.str[1] + "_" + parts.str[2]
    df["metric"] = parts.str[3:].str.join("_").str.lower()
    return df


def make_heatmap_keep_lr(partial_r_results, scenario_name="NODDI", scenario_etiv=False, split_mode="avg"):
    dfA = keep_lr(partial_r_results[(scenario_name, scenario_etiv, "A")].copy())
    dfB = keep_lr(partial_r_results[(scenario_name, scenario_etiv, "B")].copy())

    # average within each split first
    dfA_avg = dfA.groupby(["bundle", "metric"], as_index=False)["partial_r"].mean()
    dfB_avg = dfB.groupby(["bundle", "metric"], as_index=False)["partial_r"].mean()

    if split_mode == "A":
        out_df = dfA_avg.rename(columns={"partial_r": "partial_r_value"})
    elif split_mode == "B":
        out_df = dfB_avg.rename(columns={"partial_r": "partial_r_value"})
    elif split_mode == "avg":
        out_df = (
            pd.concat([dfA_avg, dfB_avg], ignore_index=True)
            .groupby(["bundle", "metric"], as_index=False)["partial_r"]
            .mean()
            .rename(columns={"partial_r": "partial_r_value"})
        )
    else:
        raise ValueError(f"Unknown split_mode: {split_mode}. Use 'avg', 'A', or 'B'.")

    heatmap = out_df.pivot_table(
        index="bundle",
        columns="metric",
        values="partial_r_value",
        aggfunc="mean"
    )

    return heatmap


def run_pca_all(heatmap, n_components=None):
    X = heatmap.values.astype(float)
    Xz = StandardScaler().fit_transform(X)

    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(Xz)

    explained = pca.explained_variance_ratio_
    loadings = pd.DataFrame(
        pca.components_.T,
        index=heatmap.columns,
        columns=[f"PC{i}" for i in range(1, pca.n_components_ + 1)]
    )
    bundle_scores = pd.DataFrame(
        scores,
        index=heatmap.index,
        columns=[f"PC{i}" for i in range(1, pca.n_components_ + 1)]
    )

    return explained, loadings, bundle_scores

In [19]:
FIG3_DIR = ROOT_DIR / "output_data" / "results" / "main_figures" / "figure3"
SUPPFIG4_DIR = ROOT_DIR / "output_data" / "results" / "supplemental_figures" / "suppfig4"
for d in [FIG3_DIR, SUPPFIG4_DIR]:
    d.mkdir(parents=True, exist_ok=True)

scenarios = [("NODDI", False), ("DKI", False)]
split_modes = ["avg", "A", "B"]

for scenario_name, scenario_etiv in scenarios:
    print(f"\n[SCENARIO] {scenario_name} | eTIV={scenario_etiv}")

    for split_mode in split_modes:
        print(f"  [SPLIT_MODE] {split_mode}")

        heatmap_keep = make_heatmap_keep_lr(
            partial_r_results,
            scenario_name,
            scenario_etiv,
            split_mode=split_mode
        )
        explained, loadings, bundle_scores = run_pca_all(heatmap_keep, n_components=2)

        pc_scores_df = bundle_scores[["PC1", "PC2"]].reset_index()
        pc_loadings_df = loadings[["PC1", "PC2"]].reset_index().rename(columns={"index": "metric"})
        explained_df = pd.DataFrame({
            "PC": ["PC1", "PC2"],
            "explained_var_ratio": explained[:2]
        })
        print(explained_df)

        if scenario_name == "NODDI" and scenario_etiv is False:
            save_dir = FIG3_DIR
            out_base = f"pca_ALL_LRkept_{split_mode}"
        elif scenario_name == "DKI" and scenario_etiv is False:
            save_dir = SUPPFIG4_DIR
            out_base = f"pca_ALL_LRkept_dki_{split_mode}"
        else:
            continue

        pc_scores_path = save_dir / f"{out_base}_bundle_scores_PC12.csv"
        pc_loadings_path = save_dir / f"{out_base}_metric_loadings_PC12.csv"
        explained_path = save_dir / f"{out_base}_explained_var_PC12.csv"

        pc_scores_df.to_csv(pc_scores_path, index=False)
        pc_loadings_df.to_csv(pc_loadings_path, index=False)
        explained_df.to_csv(explained_path, index=False)

        print(f"[SAVE] → {pc_scores_path}")
        print(f"[SAVE] → {pc_loadings_path}")
        print(f"[SAVE] → {explained_path}")


[SCENARIO] NODDI | eTIV=False
  [SPLIT_MODE] avg
    PC  explained_var_ratio
0  PC1             0.343088
1  PC2             0.222878
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/pca_ALL_LRkept_avg_bundle_scores_PC12.csv
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/pca_ALL_LRkept_avg_metric_loadings_PC12.csv
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/pca_ALL_LRkept_avg_explained_var_PC12.csv
  [SPLIT_MODE] A
    PC  explained_var_ratio
0  PC1             0.354412
1  PC2             0.220650
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/figure3/pca_ALL_LRkept_A_bundle_scores_PC12.csv
[SAVE] → /mnt/isilon/bgdlab_hbcd/projects/macedo_wm_exposome/macedo_wm_exposome/output_data/results/main_figures/